# Preprocessing

Notebook used to draft dataframeBuilerPAQ output \
Based on `preprocessing_pyod`, `preprocessing_per_statement_type` & `preprocessing_per_question`

In [1]:
import pyodbc
import pandas as pd
from sklearn.preprocessing import LabelEncoder
pd.set_option("display.max_columns", None)

In [2]:
df_test = pd.read_csv('../../../decoded_data/PAQ/FactTest.csv')
df_question = pd.read_csv('../../../decoded_data/PAQ/FactQuestionPAQ.csv')

In [3]:
df_test

,Test,CandidateKey,CreatedDateKey,VersionNumber,TestKey
0,PAQ,38114181,20190123,v1.0,166962
1,PAQ,38046331,20190110,v1.0,166963
2,PAQ,38114191,20190128,v1.0,166964
3,PAQ,38201141,20190206,v1.0,166965
4,PAQ,38337341,20190227,v1.0,166966
...,...,...,...,...,...
16675,PAQ,45444755,20240412,v1.0,183637
16676,PAQ,45512575,20240424,v1.0,183638
16677,PAQ,45579565,20240525,v1.0,183639
16678,PAQ,45638005,20240702,v1.0,183640


In [4]:
len(df_question[df_question["RightStatement"] == "ZZZ_0"])

1351765

In [5]:
len(df_question[df_question["RightStatement"] == "ZZZ_0"])

1351765

In [6]:
df_question = df_question[df_question["RightStatement"] != "ZZZ_0"]
df_question

,QuestionKey,InstanceID,LeftStatement,RightStatement,AnswerVal,Test,TestKey
0,1,1,BEW_1,BEH_1,1,PAQ,166962
1,2,1,REA_1,OPT_1,5,PAQ,166962
2,3,1,REG_1,AFW_1,5,PAQ,166962
3,4,1,RUS_1,ENE_1,5,PAQ,166962
4,5,1,OVE_1,GED_1,5,PAQ,166962
...,...,...,...,...,...,...,...
2852194,2852195,5,REA_6,OPT_6,1,PAQ,183641
2852195,2852196,5,STA_6,AMB_6,2,PAQ,183641
2852196,2852197,5,FLE_6,PLA_6,3,PAQ,183641
2852197,2852198,5,ZIC_6,AND_6,4,PAQ,183641


In [7]:
df_test["VersionNumber"].value_counts()

VersionNumber
v1.0    16680
Name: count, dtype: int64

In [8]:
df_test.drop(columns=['VersionNumber', 'Test'], inplace=True)
df_test.head()

,CandidateKey,CreatedDateKey,TestKey
0,38114181,20190123,166962
1,38046331,20190110,166963
2,38114191,20190128,166964
3,38201141,20190206,166965
4,38337341,20190227,166966


In [9]:
dropable_columns_question = ['Test']

# Future proof code (as InstanceID seems to be a mistake)
if df_question.columns.__contains__('InstanceID'):
    dropable_columns_question.append('InstanceID')

df_question.drop(columns=dropable_columns_question, inplace=True)
df_question.head()

,QuestionKey,LeftStatement,RightStatement,AnswerVal,TestKey
0,1,BEW_1,BEH_1,1,166962
1,2,REA_1,OPT_1,5,166962
2,3,REG_1,AFW_1,5,166962
3,4,RUS_1,ENE_1,5,166962
4,5,OVE_1,GED_1,5,166962


In [10]:
df = df_question.merge(df_test, on='TestKey')
df.head()

,QuestionKey,LeftStatement,RightStatement,AnswerVal,TestKey,CandidateKey,CreatedDateKey
0,1,BEW_1,BEH_1,1,166962,38114181,20190123
1,2,REA_1,OPT_1,5,166962,38114181,20190123
2,3,REG_1,AFW_1,5,166962,38114181,20190123
3,4,RUS_1,ENE_1,5,166962,38114181,20190123
4,5,OVE_1,GED_1,5,166962,38114181,20190123


In [11]:
df["AnswerVal"].value_counts()

AnswerVal
3    402852
2    377319
4    353454
1    189087
5    177793
0        10
Name: count, dtype: int64

In [12]:
df[df["AnswerVal"] == 0]

,QuestionKey,LeftStatement,RightStatement,AnswerVal,TestKey,CandidateKey,CreatedDateKey
1040976,1978477,AHZ_6,EHA_2,0,178532,43945175,20210606
1040980,1978481,EHA_2,CHA_2,0,178532,43945175,20210606
1040984,1978485,EXZ_2,AHZ_8,0,178532,43945175,20210606
1040986,1978487,DHA_10,AHA_6,0,178532,43945175,20210606
1040998,1978651,EZA_2,BZA_2,0,178533,43957755,20210616
1041003,1978656,EPZ_2,BHZ_8,0,178533,43957755,20210616
1041202,1979169,DZA_2,BXZ_8,0,178536,44027515,20210729
1119532,2128273,AHZ_4,EHA_4,0,179408,44257875,20220102
1119535,2128276,EZA_4,BHA_2,0,179408,44257875,20220102
1119540,2128281,EZA_2,BPZ_8,0,179408,44257875,20220102


In [13]:
left_values = set(df["LeftStatement"].unique())
right_values = set(df["RightStatement"].unique())
print(sorted(left_values)), print(sorted(right_values))

['AHA_2', 'AHA_4', 'AHA_8', 'AHZ_10', 'AHZ_2', 'AHZ_4', 'AHZ_6', 'AHZ_8', 'APZ_6', 'APZ_8', 'AZA_10', 'AZA_4', 'BEW_1', 'BEW_2', 'BEW_3', 'BEW_4', 'BEW_5', 'BEW_6', 'BHA_10', 'BHZ_2', 'BHZ_8', 'BPZ_4', 'BPZ_6', 'BPZ_8', 'BXZ_4', 'BXZ_8', 'CHA_10', 'CPZ_8', 'CXZ_10', 'CZA_10', 'CZA_8', 'DEN_1', 'DEN_2', 'DEN_3', 'DEN_4', 'DEN_5', 'DEN_6', 'DHA_10', 'DHA_2', 'DHA_4', 'DHZ_2', 'DOB_1', 'DOB_2', 'DOB_3', 'DOB_4', 'DOB_5', 'DOB_6', 'DXZ_8', 'DZA_10', 'DZA_2', 'DZA_4', 'DZA_6', 'DZA_8', 'EHA_10', 'EHA_2', 'EHA_4', 'EHA_8', 'EPZ_2', 'EXZ_2', 'EXZ_4', 'EZA_10', 'EZA_2', 'EZA_4', 'EZA_6', 'EZA_8', 'FLE_1', 'FLE_2', 'FLE_3', 'FLE_4', 'FLE_5', 'FLE_6', 'INT_1', 'INT_2', 'INT_3', 'INT_4', 'INT_5', 'INT_6', 'OVE_1', 'OVE_2', 'OVE_3', 'OVE_4', 'OVE_5', 'OVE_6', 'PLI_1', 'PLI_2', 'PLI_3', 'PLI_4', 'PLI_5', 'PLI_6', 'REA_1', 'REA_2', 'REA_3', 'REA_4', 'REA_5', 'REA_6', 'REG_1', 'REG_2', 'REG_3', 'REG_4', 'REG_5', 'REG_6', 'RUS_1', 'RUS_2', 'RUS_3', 'RUS_4', 'RUS_5', 'RUS_6', 'SAL_1', 'SAL_2', 'SAL_3',

(None, None)

In [14]:
df[["LeftAbbreviation", "LeftNr"]] =  df["LeftStatement"].str.split(pat="_", expand=True)
df[["RightAbbreviation", "RightNr"]] =  df["RightStatement"].str.split(pat="_", expand=True)
df['LeftRightStatement'] = df["LeftStatement"] + df["RightStatement"]
df

,QuestionKey,LeftStatement,RightStatement,AnswerVal,TestKey,CandidateKey,CreatedDateKey,LeftAbbreviation,LeftNr,RightAbbreviation,RightNr,LeftRightStatement
0,1,BEW_1,BEH_1,1,166962,38114181,20190123,BEW,1,BEH,1,BEW_1BEH_1
1,2,REA_1,OPT_1,5,166962,38114181,20190123,REA,1,OPT,1,REA_1OPT_1
2,3,REG_1,AFW_1,5,166962,38114181,20190123,REG,1,AFW,1,REG_1AFW_1
3,4,RUS_1,ENE_1,5,166962,38114181,20190123,RUS,1,ENE,1,RUS_1ENE_1
4,5,OVE_1,GED_1,5,166962,38114181,20190123,OVE,1,GED,1,OVE_1GED_1
...,...,...,...,...,...,...,...,...,...,...,...,...
1500510,2852195,REA_6,OPT_6,1,183641,45646975,20240902,REA,6,OPT,6,REA_6OPT_6
1500511,2852196,STA_6,AMB_6,2,183641,45646975,20240902,STA,6,AMB,6,STA_6AMB_6
1500512,2852197,FLE_6,PLA_6,3,183641,45646975,20240902,FLE,6,PLA,6,FLE_6PLA_6
1500513,2852198,ZIC_6,AND_6,4,183641,45646975,20240902,ZIC,6,AND,6,ZIC_6AND_6


In [15]:
left_abbreviations = set(df["LeftAbbreviation"].unique())
right_abbreviations = set(df["RightAbbreviation"].unique())
print(sorted(left_abbreviations)), print(sorted(right_abbreviations))

['AHA', 'AHZ', 'APZ', 'AZA', 'BEW', 'BHA', 'BHZ', 'BPZ', 'BXZ', 'CHA', 'CPZ', 'CXZ', 'CZA', 'DEN', 'DHA', 'DHZ', 'DOB', 'DXZ', 'DZA', 'EHA', 'EPZ', 'EXZ', 'EZA', 'FLE', 'INT', 'OVE', 'PLI', 'REA', 'REG', 'RUS', 'SAL', 'STA', 'VOL', 'ZEL', 'ZIC']
['AFW', 'AHA', 'AHZ', 'AMB', 'AND', 'APZ', 'AXZ', 'BEH', 'BHA', 'BHZ', 'BPZ', 'BXZ', 'BZA', 'CHA', 'CPZ', 'CXZ', 'CZA', 'DHA', 'DOE', 'DPZ', 'DXZ', 'DZA', 'EHA', 'EHZ', 'ENE', 'EPZ', 'EXT', 'EXZ', 'EZA', 'GED', 'INB', 'LEI', 'OPT', 'PLA', 'SAM', 'SBE', 'ZFS']


(None, None)

In [16]:
print(set.intersection(left_abbreviations, right_abbreviations))

{'CZA', 'CXZ', 'DXZ', 'EXZ', 'AHZ', 'BXZ', 'DHA', 'AHA', 'CPZ', 'BPZ', 'CHA', 'BHZ', 'BHA', 'DZA', 'EHA', 'EZA', 'EPZ', 'APZ'}


In [17]:
df_diff_statement_nrs = df[df["LeftNr"] != df["RightNr"]]
df_diff_statement_nrs

,QuestionKey,LeftStatement,RightStatement,AnswerVal,TestKey,CandidateKey,CreatedDateKey,LeftAbbreviation,LeftNr,RightAbbreviation,RightNr,LeftRightStatement
1040970,1978471,EHA_8,DHA_4,2,178532,43945175,20210606,EHA,8,DHA,4,EHA_8DHA_4
1040971,1978472,AZA_10,DZA_2,2,178532,43945175,20210606,AZA,10,DZA,2,AZA_10DZA_2
1040972,1978473,EZA_6,EXZ_10,1,178532,43945175,20210606,EZA,6,EXZ,10,EZA_6EXZ_10
1040973,1978474,EHA_10,BHA_2,2,178532,43945175,20210606,EHA,10,BHA,2,EHA_10BHA_2
1040974,1978475,AHZ_8,CHA_10,2,178532,43945175,20210606,AHZ,8,CHA,10,AHZ_8CHA_10
...,...,...,...,...,...,...,...,...,...,...,...,...
1119540,2128281,EZA_2,BPZ_8,0,179408,44257875,20220102,EZA,2,BPZ,8,EZA_2BPZ_8
1119541,2128282,DHA_2,CPZ_8,1,179408,44257875,20220102,DHA,2,CPZ,8,DHA_2CPZ_8
1119542,2128283,DZA_10,APZ_8,1,179408,44257875,20220102,DZA,10,APZ,8,DZA_10APZ_8
1119543,2128284,CZA_10,EHA_4,2,179408,44257875,20220102,CZA,10,EHA,4,CZA_10EHA_4


In [18]:
int(df_diff_statement_nrs["CreatedDateKey"].min()), int(df_diff_statement_nrs["CreatedDateKey"].max())

(20210606, 20220102)

In [19]:
df["LeftAbbreviation"].value_counts()

LeftAbbreviation
OVE    100031
INT    100031
RUS    100030
PLI    100029
ZEL    100029
BEW    100028
REA    100028
SAL    100028
STA    100028
DEN    100028
VOL    100027
ZIC    100027
REG    100026
FLE    100026
DOB    100024
DZA        18
EZA        11
EHA         8
AHZ         8
CZA         6
DHA         6
BPZ         5
BHZ         5
AZA         4
DXZ         4
CHA         3
EXZ         3
AHA         3
BXZ         3
APZ         2
BHA         2
EPZ         1
CXZ         1
CPZ         1
DHZ         1
Name: count, dtype: int64

In [20]:
pd.DataFrame([df["LeftAbbreviation"].value_counts(), df["RightAbbreviation"].value_counts()]).fillna(0).astype(int)

,OVE,INT,RUS,PLI,ZEL,BEW,REA,SAL,STA,DEN,VOL,ZIC,REG,FLE,DOB,DZA,EZA,EHA,AHZ,CZA,DHA,BPZ,BHZ,AZA,DXZ,CHA,EXZ,AHA,BXZ,APZ,BHA,EPZ,CXZ,CPZ,DHZ,EXT,GED,ENE,SAM,ZFS,DOE,AMB,BEH,SBE,OPT,LEI,AND,PLA,AFW,INB,BZA,DPZ,AXZ,EHZ
count,100031,100031,100030,100029,100029,100028,100028,100028,100028,100028,100027,100027,100026,100026,100024,18,11,8,8,6,6,5,5,4,4,3,3,3,3,2,2,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
count,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,3,15,4,8,9,3,1,0,3,3,1,6,1,1,8,2,2,3,0,100031,100031,100030,100029,100029,100028,100028,100028,100028,100028,100027,100027,100026,100026,100024,3,3,1,1


In [21]:
len(pd.DataFrame([df["LeftAbbreviation"].value_counts(), df["RightAbbreviation"].value_counts()]).fillna(0).astype(int).columns)

54

In [22]:
df["RightAbbreviation"].value_counts()

RightAbbreviation
EXT    100031
GED    100031
ENE    100030
SAM    100029
ZFS    100029
DOE    100028
AMB    100028
BEH    100028
SBE    100028
OPT    100028
LEI    100027
AND    100027
PLA    100026
AFW    100026
INB    100024
EHA        15
DZA        14
DHA         9
CZA         8
BHA         8
AHA         6
AHZ         4
BZA         3
BPZ         3
DXZ         3
CHA         3
DPZ         3
EZA         3
CPZ         3
EPZ         2
CXZ         2
BHZ         1
EXZ         1
AXZ         1
EHZ         1
BXZ         1
APZ         1
Name: count, dtype: int64

In [23]:
df_group = pd.DataFrame(df.groupby(["TestKey", "LeftStatement"]).count())
df_group.agg(['max'])

,QuestionKey,RightStatement,AnswerVal,CandidateKey,CreatedDateKey,LeftAbbreviation,LeftNr,RightAbbreviation,RightNr,LeftRightStatement
max,4,4,4,4,4,4,4,4,4,4


In [24]:
len(df), len(df_group)

(1500515, 1500489)

In [25]:
df_group["idx"] = ['_'.join(map(str,i)) for i in df_group.index.to_flat_index()]
df_group.index = df_group["idx"]
df_group.sort_values(by="QuestionKey")

,QuestionKey,RightStatement,AnswerVal,CandidateKey,CreatedDateKey,LeftAbbreviation,LeftNr,RightAbbreviation,RightNr,LeftRightStatement,idx
idx,,,,,,,,,,,
166962_BEW_1,1,1,1,1,1,1,1,1,1,1,166962_BEW_1
178080_INT_5,1,1,1,1,1,1,1,1,1,1,178080_INT_5
178080_INT_4,1,1,1,1,1,1,1,1,1,1,178080_INT_4
178080_INT_3,1,1,1,1,1,1,1,1,1,1,178080_INT_3
178080_INT_2,1,1,1,1,1,1,1,1,1,1,178080_INT_2
...,...,...,...,...,...,...,...,...,...,...,...
179408_DZA_10,3,3,3,3,3,3,3,3,3,3,179408_DZA_10
178536_DXZ_8,3,3,3,3,3,3,3,3,3,3,178536_DXZ_8
179407_DZA_4,4,4,4,4,4,4,4,4,4,4,179407_DZA_4


In [26]:
len(df["LeftRightStatement"].unique())

184